# LangChain: Q&A over Documents (Groq Llama 3.1)

## Overview
Learn how to build a question-answering system over your own documents using:
- **Document Loaders** - Load data from various sources
- **Embeddings** - Convert text to numerical vectors
- **Vector Stores** - Store and search embeddings
- **Retrieval QA Chain** - Combine retrieval + LLM for answers

**Model Used:** Groq Llama-3.1-8b-instant  
**Embeddings:** HuggingFace (sentence-transformers)

**Use Case:** Query a product catalog, company docs, or any text corpus


In [ ]:
# Cell 2: Install Required Packages

!pip install langchain langchain-groq python-dotenv docarray sentence-transformers -q

In [ ]:
# Cell 3: Import Libraries and Load Environment

import os
import warnings
from dotenv import load_dotenv, find_dotenv
from IPython.display import display, Markdown

# Load environment variables
load_dotenv(find_dotenv())

# Suppress warnings
warnings.filterwarnings('ignore')

# Verify API key
groq_api_key = os.environ.get('GROQ_API_KEY')
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file!")
    
print("✅ Environment loaded successfully")


In [ ]:
# Cell 4: Initialize Groq LLM

from langchain_groq import ChatGroq

# Set model
llm_model = "llama-3.1-8b-instant"

# Initialize LLM
llm = ChatGroq(temperature=0, model=llm_model, groq_api_key=groq_api_key)

print(f"✅ Model initialized: {llm_model}")


In [ ]:
# Cell 5: Load Document Data

from langchain.document_loaders import CSVLoader

# Load the outdoor clothing catalog
file = 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)

print(f"✅ CSV Loader created for: {file}")


In [ ]:
# Cell 6: Quick Start - Create Index and Query

from langchain.indexes import VectorstoreIndexCreator
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_community.embeddings import HuggingFaceEmbeddings

# Use HuggingFace embeddings (free, local)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create vector store index in one line
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embeddings
).from_loaders([loader])

print("✅ Vector store index created successfully")


In [ ]:
# Cell 7: Run Query on Index

# Define query
query = "Please list all your shirts with sun protection in a table \
in markdown and summarize each one."

# Query the index
response = index.query(query, llm=llm)

# Display response
display(Markdown(response))


## Step-by-Step: Under the Hood

Now let's break down what's happening in the one-liner approach. We'll go through each step manually to understand the process.


In [ ]:
# Cell 8: Step-by-Step - Load Documents

from langchain.document_loaders import CSVLoader

# Load documents
loader = CSVLoader(file_path=file)
docs = loader.load()

print(f"✅ Loaded {len(docs)} documents")
print("\nFirst document:")
print(docs[0])


In [ ]:
# Cell 9: Create Embeddings

from langchain_community.embeddings import HuggingFaceEmbeddings

# Initialize embeddings model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embeddings model loaded")


In [ ]:
# Cell 10: Test Embeddings

# Create embedding for a sample text
embed = embeddings.embed_query("Hi my name is Harrison")

print(f"Embedding dimension: {len(embed)}")
print(f"\nFirst 5 values: {embed[:5]}")


In [ ]:
# Cell 11: Create Vector Database

from langchain_community.vectorstores import DocArrayInMemorySearch

# Create vector store from documents
db = DocArrayInMemorySearch.from_documents(
    docs, 
    embeddings
)

print(f"✅ Vector database created with {len(docs)} documents")


In [ ]:
# Cell 12: Test Similarity Search

# Query for similar documents
query = "Please suggest a shirt with sunblocking"
docs = db.similarity_search(query)

print(f"Found {len(docs)} similar documents")
print("\nMost similar document:")
print(docs[0])


In [ ]:
# Cell 13: Create Retriever

# Convert vector store to retriever
retriever = db.as_retriever()

print("✅ Retriever created from vector store")


In [ ]:
# Cell 14: Manual Question Answering (Without Chain)

# Combine retrieved documents into single text
qdocs = "".join([docs[i].page_content for i in range(len(docs))])

# Create prompt with documents + question
response = llm.invoke(
    f"{qdocs}\n\nQuestion: Please list all your shirts with sun protection \
in a table in markdown and summarize each one."
)

# Display response
display(Markdown(response.content))


## RetrievalQA Chain

Instead of manually combining documents and querying the LLM, we can use the **RetrievalQA** chain to automate this process.

**Chain Types:**
- **stuff** - Put all docs into one prompt (simplest, works for small docs)
- **map_reduce** - Query each doc separately, then combine answers
- **refine** - Iteratively build answer across docs
- **map_rerank** - Score each doc's answer, return highest


In [ ]:
# Cell 16: Create RetrievalQA Chain

from langchain.chains import RetrievalQA

# Create QA chain with "stuff" method
qa_stuff = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff",  # Simple method: stuff all docs into one prompt
    retriever=retriever, 
    verbose=True
)

print("✅ RetrievalQA chain created")


In [ ]:
# Cell 17: Run RetrievalQA Chain

# Define query
query = "Please list all your shirts with sun protection in a table \
in markdown and summarize each one."

# Run the chain
response = qa_stuff.run(query)

# Display response
display(Markdown(response))


In [ ]:
# Cell 18: Compare Index Query vs RetrievalQA

# Both approaches produce similar results
print("Method 1: Index Query (one-liner)")
response1 = index.query(query, llm=llm)

print("\nMethod 2: RetrievalQA Chain (step-by-step)")
response2 = qa_stuff.run(query)

print("\n✅ Both methods achieve the same result!")
print("   - Index Query: Fast prototyping")
print("   - RetrievalQA: More control and customization")


In [ ]:
# Cell 19: Advanced - Custom Embeddings in Index

# You can customize the index creation with specific embeddings
custom_index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
).from_loaders([loader])

print("✅ Custom index created with specified embeddings")


In [ ]:
# Cell 20: Test Different Queries

# Try different questions about the catalog
queries = [
    "What products are available for cold weather?",
    "Do you have any jackets with waterproof features?",
    "What's the price range of your products?"
]

for q in queries:
    print(f"\nQuery: {q}")
    print("-" * 60)
    response = qa_stuff.run(q)
    print(response)
    print()


## Chain Types Comparison

### 1. **Stuff** (Used in this notebook)
- **How it works:** Combines all retrieved docs into one prompt
- **Pros:** Simple, cheap (one LLM call), works well for small docs
- **Cons:** Limited by LLM context window
- **Best for:** Small to medium document sets

### 2. **Map Reduce**
- **How it works:** Process each doc separately, then combine results
- **Pros:** Works with any number of docs, can parallelize
- **Cons:** More expensive (multiple LLM calls), treats docs independently
- **Best for:** Large document sets, parallel processing

### 3. **Refine**
- **How it works:** Iteratively build answer by processing docs sequentially
- **Pros:** Good for building comprehensive answers over time
- **Cons:** Slow (sequential), expensive, can be verbose
- **Best for:** Detailed, evolving answers

### 4. **Map Rerank**
- **How it works:** Score each doc's relevance, return best answer
- **Pros:** Fast (parallel), focused answers
- **Cons:** Requires good scoring prompts, multiple LLM calls
- **Best for:** Finding single best answer from many docs


In [ ]:
# Cell 22: Try Map Reduce Chain Type

# Create QA chain with map_reduce
qa_map_reduce = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="map_reduce",
    retriever=retriever, 
    verbose=True
)

# Test query
query = "Summarize all the jacket options available"
response = qa_map_reduce.run(query)

print("\nMap Reduce Response:")
display(Markdown(response))


## Summary: Q&A over Documents

### Key Components:

1. **Document Loaders** - Import data (CSV, PDF, web, etc.)
2. **Embeddings** - Convert text to numerical vectors (semantic meaning)
3. **Vector Store** - Store embeddings for similarity search
4. **Retriever** - Interface to fetch relevant documents
5. **RetrievalQA Chain** - Combine retrieval + LLM for answers

### Workflow:
Documents → Chunks → Embeddings → Vector Store
↓
Query → Embedding → Similarity Search → Retrieve Docs → LLM → Answer


### Chain Types:

| Chain Type | Speed | Cost | Best For |
|------------|-------|------|----------|
| **stuff** | Fast | Low | Small docs |
| **map_reduce** | Medium | High | Large docs, parallel |
| **refine** | Slow | High | Detailed answers |
| **map_rerank** | Medium | High | Best single answer |

### When to Use:

- **Internal knowledge bases** - Company wikis, documentation
- **Customer support** - Product manuals, FAQs
- **Research** - Academic papers, reports
- **E-commerce** - Product catalogs (like this example)